<a href="https://colab.research.google.com/github/yogeshwardev/csa6301_Thread_intelligence_network_security/blob/main/lap_programs_output_colab_from_24_to_32.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
def rule_matches(packet, rule):
    def field_ok(value, rule_value):
        return rule_value == "any" or value == rule_value

    return (
        field_ok(packet["src"], rule["src"])
        and field_ok(packet["dst"], rule["dst"])
        and field_ok(packet["port"], rule["port"])
        and field_ok(packet["proto"], rule["proto"])
    )


def evaluate_packet(packet, rules):
    """Return the action of the first matching rule (top-down), or 'deny' if no rule matches (default-deny)."""
    for rule in rules:
        if rule_matches(packet, rule):
            return rule["action"]
    return "deny"


# Test Cases
def test_experiment24():
    misordered_rules = [
        {"action": "allow", "src": "any", "dst": "any", "port": "any", "proto": "any"},
        {"action": "deny", "src": "10.0.0.5", "dst": "any", "port": "any", "proto": "any"},
    ]
    packet = {"src": "10.0.0.5", "dst": "8.8.8.8", "port": 443, "proto": "tcp"}
    assert evaluate_packet(packet, misordered_rules) == "allow"

    fixed_rules = [
        {"action": "deny", "src": "10.0.0.5", "dst": "any", "port": "any", "proto": "any"},
        {"action": "allow", "src": "any", "dst": "any", "port": "any", "proto": "any"},
    ]
    assert evaluate_packet(packet, fixed_rules) == "deny"

    other_packet = {"src": "192.168.1.9", "dst": "1.1.1.1", "port": 53, "proto": "udp"}
    assert evaluate_packet(other_packet, []) == "deny"
    print("Experiment 24: All test cases passed.")


test_experiment24()


Experiment 24: All test cases passed.


In [2]:
def handle_outbound_syn(packet, state_table, allow_rules):
    """packet: {"src","sport","dst","dport"} outbound SYN.
    If (dst, dport) is permitted by allow_rules, record state so the
    matching return traffic will be permitted automatically."""
    key = (packet["dst"], packet["dport"])
    if key in allow_rules:
        state_table[(packet["src"], packet["sport"], packet["dst"], packet["dport"])] = "ESTABLISHED"
        return "allow"
    return "deny"


def handle_inbound_packet(packet, state_table):
    """packet: {"src","sport","dst","dport"} inbound packet claiming to be
    a response. Only allowed if it matches an existing state-table entry
    for the original outbound connection (reversed direction)."""
    key = (packet["dst"], packet["dport"], packet["src"], packet["sport"])
    if key in state_table:
        return "allow"
    return "deny"


# Test Cases
def test_experiment25():
    state_table = {}
    allow_rules = {("93.184.216.34", 443)}
    outbound = {"src": "10.0.0.20", "sport": 51000, "dst": "93.184.216.34", "dport": 443}
    assert handle_outbound_syn(outbound, state_table, allow_rules) == "allow"
    assert (outbound["src"], outbound["sport"], outbound["dst"], outbound["dport"]) in state_table

    response = {"src": "93.184.216.34", "sport": 443, "dst": "10.0.0.20", "dport": 51000}
    assert handle_inbound_packet(response, state_table) == "allow"

    unsolicited = {"src": "203.0.113.9", "sport": 4444, "dst": "10.0.0.20", "dport": 51000}
    assert handle_inbound_packet(unsolicited, state_table) == "deny"
    print("Experiment 25: All test cases passed.")


test_experiment25()


Experiment 25: All test cases passed.


In [3]:
IDS_RULES = [
    {
        "sid": 2000001,
        "msg": "Possible SQL Injection",
        "proto": "tcp",
        "dst_port": 80,
        "content": "union select",
    },
    {
        "sid": 2000002,
        "msg": "Directory Traversal Attempt",
        "proto": "tcp",
        "dst_port": 80,
        "content": "../../../etc/passwd",
    },
    {
        "sid": 2000003,
        "msg": "Suspicious RDP Brute Force Pattern",
        "proto": "tcp",
        "dst_port": 3389,
        "content": "login_attempt",
    },
]


def scan_packet(packet, rules=IDS_RULES):
    """packet: {"proto","dst_port","payload"}. A rule fires only if BOTH
    the protocol/port match AND the content pattern is found (case-insensitive)."""
    alerts = []
    payload_lower = packet["payload"].lower()
    for rule in rules:
        if (
            packet["proto"] == rule["proto"]
            and packet["dst_port"] == rule["dst_port"]
        ):
            if rule["content"] in payload_lower:
                alerts.append(rule["msg"])
    return alerts


# Test Cases
def test_experiment26():
    sqli_packet = {
        "proto": "tcp",
        "dst_port": 80,
        "payload": "id=1' UNION SELECT user,pass FROM accounts--",
    }
    assert "Possible SQL Injection" in scan_packet(sqli_packet)
    traversal_packet = {
        "proto": "tcp",
        "dst_port": 80,
        "payload": "GET /files?path=../../../etc/passwd",
    }
    assert "Directory Traversal Attempt" in scan_packet(traversal_packet)

    wrong_context = {
        "proto": "tcp",
        "dst_port": 22,
        "payload": "id=1' UNION SELECT user,pass FROM accounts--",
    }
    assert scan_packet(wrong_context) == []
    print("Experiment 26: All test cases passed.")


test_experiment26()


Experiment 26: All test cases passed.


In [4]:
IDS_RULES = [
    {
        "sid": 2000001,
        "msg": "Possible SQL Injection",
        "proto": "tcp",
        "dst_port": 80,
        "content": "union select",
    },
    {
        "sid": 2000002,
        "msg": "Directory Traversal Attempt",
        "proto": "tcp",
        "dst_port": 80,
        "content": "../../../etc/passwd",
    },
    {
        "sid": 2000003,
        "msg": "Suspicious RDP Brute Force Pattern",
        "proto": "tcp",
        "dst_port": 3389,
        "content": "login_attempt",
    },
]


def scan_packet(packet, rules=IDS_RULES):
    alerts = []
    payload_lower = packet["payload"].lower()
    for rule in rules:
        if (
            packet["proto"] == rule["proto"]
            and packet["dst_port"] == rule["dst_port"]
        ):
            if rule["content"] in payload_lower:
                alerts.append(rule["msg"])
    return alerts


def ips_process(packet, rules, whitelist=None):
    """Decides whether to BLOCK (untrusted source) or ALLOW-WITH-LOG (source on tuning whitelist)."""
    whitelist = whitelist or set()
    alerts = scan_packet(packet, rules)
    if not alerts:
        return {"action": "allow", "alerts": []}
    if packet.get("src_ip") in whitelist:
        return {
            "action": "allow",
            "alerts": alerts,
            "note": "source whitelisted, alert suppressed from blocking",
        }
    return {"action": "block", "alerts": alerts}


# Test Cases
def test_experiment27():
    attack_packet = {
        "proto": "tcp",
        "dst_port": 80,
        "payload": "union select username,password from users",
        "src_ip": "203.0.113.50",
    }
    result = ips_process(attack_packet, IDS_RULES)
    assert result["action"] == "block"

    partner_packet = dict(attack_packet, src_ip="198.51.100.10")
    result2 = ips_process(partner_packet, IDS_RULES, whitelist={"198.51.100.10"})
    assert result2["action"] == "allow"
    assert "note" in result2
    print("Experiment 27: All test cases passed.")


test_experiment27()


Experiment 27: All test cases passed.


In [5]:
def xor_cipher(data: bytes, key: bytes) -> bytes:
    return bytes(b ^ key[i % len(key)] for i, b in enumerate(data))


def encrypt_tunnel(plaintext: str, key: str) -> bytes:
    return xor_cipher(plaintext.encode(), key.encode())


def decrypt_tunnel(ciphertext: bytes, key: str) -> str:
    return xor_cipher(ciphertext, key.encode()).decode(errors="replace")


# Test Cases
def test_experiment28():
    key = "VPN-Shared-Secret-Key"
    plaintext = "Transfer $50,000 to account 998211 - confidential"
    ciphertext = encrypt_tunnel(plaintext, key)

    assert ciphertext != plaintext.encode()
    assert (
        b"Transfer" not in ciphertext
    ), "Sensitive plaintext must not be visible in captured tunnel data"

    recovered = decrypt_tunnel(ciphertext, key)
    assert recovered == plaintext

    wrong_recovered = decrypt_tunnel(ciphertext, "Wrong-Key")
    assert wrong_recovered != plaintext
    print("Experiment 28: All test cases passed.")


test_experiment28()


Experiment 28: All test cases passed.


In [6]:
SEGMENTATION_POLICY = {
    ("Guest", "Guest"): True,
    ("Guest", "Corporate"): False,
    ("Guest", "Medical"): False,
    ("Corporate", "Corporate"): True,
    ("Corporate", "Medical"): False,
    ("Admin", "Medical"): True,
    ("Admin", "Corporate"): True,
    ("Admin", "Admin"): True,
}


def can_communicate(src_segment, dst_segment, policy=SEGMENTATION_POLICY):
    """Fail-safe default: any undefined segment pair is denied."""
    return policy.get((src_segment, dst_segment), False)


# Test Cases
def test_experiment29():
    assert can_communicate("Guest", "Medical") is False
    assert can_communicate("Guest", "Corporate") is False
    assert can_communicate("Admin", "Medical") is True
    assert can_communicate("Admin", "Corporate") is True
    assert can_communicate("Corporate", "Corporate") is True
    assert can_communicate("IoT", "Corporate") is False
    print("Experiment 29: All test cases passed.")


test_experiment29()


Experiment 29: All test cases passed.


In [7]:
ROLE_REQUIREMENTS = {
    "intern": {"read_reports"},
    "analyst": {"read_reports", "write_reports"},
    "admin": {"read_reports", "write_reports", "manage_users", "manage_servers"},
}


def find_over_privileged_users(users, role_requirements=ROLE_REQUIREMENTS):
    """users: {username: {"role": str, "granted_permissions": set}}.
    Returns {username: excess_permissions} for anyone holding more access
    than their role requires — a least-privilege violation."""
    violations = {}
    for username, info in users.items():
        required = role_requirements.get(info["role"], set())
        excess = info["granted_permissions"] - required
        if excess:
            violations[username] = excess
    return violations


# Test Cases
def test_experiment30():
    users = {
        "jdoe": {"role": "intern", "granted_permissions": {"read_reports"}},
        "asmith": {
            "role": "analyst",
            "granted_permissions": {"read_reports", "write_reports"},
        },
        "kintern": {
            "role": "intern",
            "granted_permissions": {
                "read_reports",
                "manage_users",
                "manage_servers",
            },
        },
    }
    violations = find_over_privileged_users(users)
    assert "jdoe" not in violations
    assert "asmith" not in violations
    assert "kintern" in violations
    assert violations["kintern"] == {"manage_users", "manage_servers"}
    print("Experiment 30: All test cases passed.")


test_experiment30()


Experiment 30: All test cases passed.


In [8]:
def zero_trust_authorize(request, resource_policy):
    """request: {"user","mfa_passed","device_posture":{"antivirus_enabled","os_patched"},"resource"}
    resource_policy: {resource: set(allowed_users)}
    Access requires: user authorized for resource AND mfa_passed AND healthy device posture."""
    reasons = []
    allowed_users = resource_policy.get(request["resource"], set())

    if request["user"] not in allowed_users:
        reasons.append("user not authorized for this resource")
    if not request["mfa_passed"]:
        reasons.append("MFA not completed")
    if not request["device_posture"]["antivirus_enabled"]:
        reasons.append("antivirus disabled")
    if not request["device_posture"]["os_patched"]:
        reasons.append("OS not fully patched")

    return {"granted": len(reasons) == 0, "reasons": reasons}


# Test Cases
def test_experiment31():
    policy = {"finance_db": {"csmith", "afinance"}}
    healthy_request = {
        "user": "csmith",
        "mfa_passed": True,
        "device_posture": {"antivirus_enabled": True, "os_patched": True},
        "resource": "finance_db",
    }
    assert zero_trust_authorize(healthy_request, policy)["granted"] is True

    # Same user, correct password/MFA, but antivirus disabled -> still denied
    unhealthy_request = dict(
        healthy_request,
        device_posture={"antivirus_enabled": False, "os_patched": True},
    )
    result2 = zero_trust_authorize(unhealthy_request, policy)
    assert result2["granted"] is False
    assert "antivirus disabled" in result2["reasons"]

    # A stolen credential for a user never authorized for this resource
    unauthorized_request = dict(healthy_request, user="attacker99")
    assert zero_trust_authorize(unauthorized_request, policy)["granted"] is False
    print("Experiment 31: All test cases passed.")


test_experiment31()


Experiment 31: All test cases passed.


In [9]:
def simulate_defense(attack, layers):
    """layers: ordered list of (layer_name, catch_function).
    Returns the name of the first layer that catches the attack, or 'BREACH' if it passes undetected."""
    for layer_name, catch_fn in layers:
        if catch_fn(attack):
            return layer_name
    return "BREACH"


def spam_filter_catches(attack):
    return attack.get("known_phishing_domain", False)


def endpoint_av_catches(attack):
    return attack.get("known_malware_hash", False)


def segmentation_catches(attack):
    return attack.get("attempts_lateral_movement", False) and not attack.get(
        "segmentation_bypassed", False
    )


def siem_monitoring_catches(attack):
    return attack.get("generates_anomalous_traffic", False)


# Test Cases
def test_experiment32():
    layers = [
        ("Email Spam Filter", spam_filter_catches),
        ("Endpoint Antivirus", endpoint_av_catches),
        ("Network Segmentation", segmentation_catches),
        ("SIEM Monitoring", siem_monitoring_catches),
    ]

    # Phishing email evades spam filter (unknown domain), but AV catches the attachment
    attack1 = {"known_phishing_domain": False, "known_malware_hash": True}
    assert simulate_defense(attack1, layers) == "Endpoint Antivirus"

    # Evades spam filter and AV, but is caught while attempting lateral movement
    attack2 = {
        "known_phishing_domain": False,
        "known_malware_hash": False,
        "attempts_lateral_movement": True,
        "segmentation_bypassed": False,
    }
    assert simulate_defense(attack2, layers) == "Network Segmentation"

    # Evades every layer entirely
    attack3 = {
        "known_phishing_domain": False,
        "known_malware_hash": False,
        "attempts_lateral_movement": False,
        "generates_anomalous_traffic": False,
    }
    assert simulate_defense(attack3, layers) == "BREACH"
    print("Experiment 32: All test cases passed.")


test_experiment32()


Experiment 32: All test cases passed.
